##### Imports and setup

In [95]:
from modules.analysis_toolkit.helpers.config.constants import CAPTURE_RATE_IC
from modules.analysis_toolkit.helpers.index_finder import GeoOptions
# setup the notebook environment
!pixi install
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys
import logging
logging.getLogger("pypsa.network.descriptors").setLevel(logging.ERROR)

def ensure_project_root_on_path(marker_dir: str = "modules") -> str:
    """Find the nearest ancestor folder containing `marker_dir`, add it to
    sys.path (front) and return its path. Raises RuntimeError if not found.
    """
    cwd = Path.cwd().resolve()
    # If cwd contains marker_dir, use cwd
    if (cwd / marker_dir).exists():
        p = str(cwd)
        if p not in sys.path:
            sys.path.insert(0, p)
        return p
    # Walk parents
    for parent in cwd.parents:
        if (parent / marker_dir).exists():
            p = str(parent)
            if p not in sys.path:
                sys.path.insert(0, p)
            return p
    raise RuntimeError(f"Could not find '{marker_dir}' directory in {cwd} or any parent")


project_root = ensure_project_root_on_path()
print("Added project root to sys.path:", project_root)

# imports and loading data
from modules.analysis_toolkit.analyzer import ResultsComputer, GeoOptions
from modules.analysis_toolkit.helpers.plotting import TimeSeriesPlot, HistogramPlot, BarChartPlot, WaterfallPlot
import pandas as pd

study_years = [2030, 2040]
rc = {year: ResultsComputer(year=year) for year in study_years}

✔ The default environment has been installed.
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Added project root to sys.path: /Users/rca/PycharmProjects/NGV-FBMC


INFO:pypsa.network.io:New version 1.1.2 available! (Current: 1.0.6)
INFO:pypsa.network.io:Imported network 'Status Quo (SQ) - 2030' has buses, carriers, generators, global_constraints, lines, links, loads, shapes, storage_units, stores, sub_networks
INFO:pypsa.network.io:New version 1.1.2 available! (Current: 1.0.6)
INFO:pypsa.network.io:Imported network 'Integrated Energy Market (IEM) - 2030' has buses, carriers, generators, global_constraints, lines, links, loads, shapes, storage_units, stores, sub_networks
INFO:pypsa.network.io:New version 1.1.2 available! (Current: 1.0.6)
INFO:pypsa.network.io:Imported network 'Flow-based market coupling (FBMC) - 2030' has buses, carriers, generators, global_constraints, lines, links, loads, shapes, storage_units, stores, sub_networks
INFO:pypsa.network.io:New version 1.1.2 available! (Current: 1.0.6)
INFO:pypsa.network.io:Imported network 'SQ (2030) - redispatch' has buses, carriers, generators, global_constraints, lines, links, loads, shapes, sto

This notebook computes the key indicators reported in [Model run validation checks](https://docs.google.com/document/d/165EH2i8Fgtec260gfUZL6zXfYESi6Uwv6g7zTVdtgiU/edit?pli=1&tab=t.0) to check the consistency of the model runs and the results. The indicators are computed for both the dispatch and redispatch results, and can be compared to the values reported in FES25 or other reference documents.

# Dispatch

## Total GB demand
As reference, FES25 uses the following total GB demand:
* 339 TWh in 2030
* 564 TWh in 2040

In [24]:
pd.concat(
    [
        rc[year].withdrawal.compare_dispatch(bus_carrier=["AC", "AC_OH"], groupby=["bus", "carrier"]) \
            .drop("Line", level="component") \
            .drop(["DC", "DC_OH"], level="carrier") \
            .query("bus.str.contains('GB ')") \
            .sum() \
            .div(1e6) \
            .round(1)
        for year in study_years
    ],
    keys=study_years,
    names=["year"],
    axis=1
)

Calling method withdrawal
Calling method withdrawal


year,2030,2040
scenario,,
sq,361.0,633.1
iem,361.0,633.2
iem_fb,360.9,630.6
diff: iem-sq,0.0,0.1
diff: iemfb-iem,-0.1,-2.6


## Total GB supply

In [25]:
pd.concat(
    [
        rc[year].supply.compare_dispatch(bus_carrier=["AC", "AC_OH"], groupby=["bus", "carrier"]) \
            .drop("Line", level="component") \
            .drop(["DC", "DC_OH"], level="carrier") \
            .query("bus.str.contains('GB ')") \
            .sum() \
            .div(1e6) \
            .round(1)
        for year in study_years
    ],
    keys=study_years,
    names=["year"],
    axis=1
)

Calling method supply
Calling method supply


year,2030,2040
scenario,,
sq,433.2,788.0
iem,433.3,788.0
iem_fb,432.2,773.4
diff: iem-sq,0.1,0.1
diff: iemfb-iem,-1.1,-14.6


## GB net position

In [26]:
pd.concat(
    [
        rc[year].energy_balance.compare_dispatch(bus_carrier=["AC", "AC_OH"], groupby=["bus", "carrier"]) \
            .drop("Line", level="component") \
            .drop(["DC", "DC_OH"], level="carrier") \
            .query("bus.str.contains('GB ')") \
            .sum() \
            .div(1e6) \
            .round(1)
        for year in study_years
    ],
    keys=study_years,
    names=["year"],
    axis=1
)

Calling method energy_balance
Calling method energy_balance


year,2030,2040
scenario,,
sq,72.2,154.8
iem,72.3,154.8
iem_fb,71.2,142.8
diff: iem-sq,0.1,-0.0
diff: iemfb-iem,-1.0,-12.0


## GB prices

In [33]:
pd.concat(
    [
        rc[year].prices.compare_dispatch(bus_carrier=["AC", "AC_OH"]) \
            .query("name.str.contains('GB ')") \
            .describe() \
            .round(1) \
        for year in study_years
    ],
    keys=study_years,
    names=["year"],
    axis=1
)

Calling method prices
Calling method prices


year      2030                                            2040               \
scenario    sq   iem iem_fb diff: iem-sq diff: iemfb-iem    sq   iem iem_fb   
count     24.0  24.0   24.0         24.0            24.0  24.0  24.0   24.0   
mean      29.1  29.7   29.0          0.6            -0.7   3.2   3.2    3.3   
std        2.3   2.3    3.0          0.8             2.7   0.4   0.3    0.3   
min       21.2  21.6   21.6         -1.1            -6.0   2.3   2.3    2.2   
25%       28.0  29.0   27.6          0.3            -2.2   3.1   3.1    3.3   
50%       28.8  29.6   28.5          0.5            -1.0   3.3   3.3    3.4   
75%       30.9  31.2   30.0          0.7             0.1   3.3   3.3    3.5   
max       32.6  33.5   34.8          2.5             5.6   4.2   3.8    3.8   

year                                   
scenario diff: iem-sq diff: iemfb-iem  
count            24.0            24.0  
mean              0.0             0.1  
std               0.1             0.3  
min              -0.4            -0.6  
25%              -0.0             0.0  
50%               0.0             0.1  
75%               0.0             0.3  
max               0.2             0.9

## Merchant congestion revenue

In [36]:
pd.concat(
    [
        rc[year].congestion_income.compare_dispatch(where=GeoOptions.GB_ONLY) \
            .groupby("scenario", axis=1).sum() \
            .div(1e6) \
            .round(1) \
        for year in study_years
    ],
    keys=study_years,
    names=["year"],
    axis=1
)

Calling method congestion_income


/var/folders/9b/zpm4mfyd03s_22tcv6xfdv780000gn/T/ipykernel_61344/3010035573.py:4: FutureWarning:

DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.



Calling method congestion_income


/var/folders/9b/zpm4mfyd03s_22tcv6xfdv780000gn/T/ipykernel_61344/3010035573.py:4: FutureWarning:

DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.



year                          2030                                       \
scenario              diff: iem-sq diff: iemfb-iem    iem iem_fb     sq   
name                                                                      
Aminth                        -3.5             0.2  533.3  533.4  536.7   
BritNed                       -4.1            -0.3  617.3  617.0  621.5   
Continental Link               0.0             0.0    0.0    0.0    0.0   
Cronos                         0.0             0.0    0.0    0.0    0.0   
East-West                     -0.5            -0.1   47.9   47.8   48.4   
ElecLink                      -2.1             1.5  447.6  449.1  449.7   
FAB Link                       0.0             0.0    0.0    0.0    0.0   
Greenlink (Greenwire)         -0.4            -0.1   42.9   42.8   43.3   
Gridlink                       0.0             0.0    0.0    0.0    0.0   
IFA                           -4.1             4.2  895.2  899.3  899.3   
IFA2                          -2.2             2.1  447.6  449.7  449.8   
Kulizumboo                     0.0             0.0    0.0    0.0    0.0   
LirIC                          0.0             0.0    0.0    0.0    0.0   
MARES                         -0.6            -0.2   61.4   61.2   62.0   
Moyle                         -0.8            -0.6   81.8   81.2   82.6   
NS Link (NSL)                  0.9            -0.6  503.0  502.4  502.1   
Nautilus                       0.0             0.0    0.0    0.0    0.0   
Nemo                          -2.9            -0.1  526.7  526.6  529.6   
NeuConnect                     0.0             0.0    0.0    0.0    0.0   
NorthConnect                   0.0             0.0    0.0    0.0    0.0   
SENECA                         0.0             0.0    0.0    0.0    0.0   
Viking Link                   -3.5             0.2  533.3  533.4  536.7   

year                          2040                                       
scenario              diff: iem-sq diff: iemfb-iem    iem iem_fb     sq  
name                                                                     
Aminth                        -1.1            -3.0  143.6  140.6  144.7  
BritNed                        0.1             1.3  287.9  289.3  287.8  
Continental Link              -0.3            -1.7   32.8   31.1   33.1  
Cronos                        -0.2             1.8  356.9  358.8  357.1  
East-West                     -0.0            -0.6   40.3   39.6   40.3  
ElecLink                      -0.8            -2.3  197.1  194.8  197.9  
FAB Link                      -0.6            -5.1  246.4  241.3  247.0  
Greenlink (Greenwire)         -0.0            -0.8   36.1   35.3   36.1  
Gridlink                      -0.6             2.5  246.4  248.9  247.0  
IFA                           -1.7             0.8  394.2  395.1  395.9  
IFA2                          -0.2            -7.0  197.1  190.1  197.3  
Kulizumboo                    -0.2            -4.8  138.0  133.2  138.2  
LirIC                         -0.1            -0.8   48.2   47.4   48.2  
MARES                         -0.1            -0.8   51.6   50.9   51.7  
Moyle                         -0.1            -0.7   63.5   62.8   63.6  
NS Link (NSL)                 -0.2            -1.5   27.0   25.5   27.2  
Nautilus                      -0.2             2.0  382.4  384.4  382.6  
Nemo                          -0.1            -0.7  254.9  254.3  255.1  
NeuConnect                    -1.1             2.1  470.8  473.0  471.9  
NorthConnect                  -0.2            -3.1   25.5   22.4   25.7  
SENECA                         0.1             1.4  300.9  302.3  300.8  
Viking Link                   -1.1            -3.0  143.6  140.6  144.7

In [43]:
pd.concat(
    [
        rc[year].congestion_income.compare_dispatch(where=GeoOptions.GB_ONLY) \
            .groupby("scenario", axis=1).sum() \
            .sum(axis=0) \
            .div(1e6) \
            .round(1) \
        for year in study_years
    ],
    keys=study_years,
    names=["year"],
    axis=1
)

Calling method congestion_income


/var/folders/9b/zpm4mfyd03s_22tcv6xfdv780000gn/T/ipykernel_61344/487377846.py:4: FutureWarning:

DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.



Calling method congestion_income


/var/folders/9b/zpm4mfyd03s_22tcv6xfdv780000gn/T/ipykernel_61344/487377846.py:4: FutureWarning:

DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.



year,2030,2040
scenario,,
diff: iem-sq,-23.6,-8.5
diff: iemfb-iem,6.1,-24.0
iem,4737.9,4085.5
iem_fb,4744.0,4061.5
sq,4761.6,4094.0


## SEW change

In [58]:
pd.concat(
    [
        rc[year].welfare_system.compare_dispatch() \
            .groupby("scenario", axis=1).sum() \
            .sum(axis=0) \
            .round(1) \
        for year in study_years
    ],
    keys=study_years,
    names=["year"],
    axis=1
)
# in M€

Calling method welfare_system


/var/folders/9b/zpm4mfyd03s_22tcv6xfdv780000gn/T/ipykernel_61344/3439889639.py:4: FutureWarning:

DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.



Calling method welfare_system


/var/folders/9b/zpm4mfyd03s_22tcv6xfdv780000gn/T/ipykernel_61344/3439889639.py:4: FutureWarning:

DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.



year,2030,2040
scenario,,
diff: iem-sq,6.6,2.5
diff: iemfb-iem,-16.4,-109.7
iem,-148686.4,-103748.4
iem_fb,-148702.8,-103858.1
sq,-148693.0,-103750.9


## Breakdown of GB generation per technology

In [51]:
pd.concat(
    [
        rc[year].supply.compare_dispatch(bus_carrier=["AC", "AC_OH"], groupby=["bus", "carrier"]) \
            .drop("Line", level="component") \
            .drop(["DC", "DC_OH"], level="carrier") \
            .query("bus.str.contains('GB ')")  \
            .groupby("carrier").sum() \
            .div(1e6) \
            .round(1)
        for year in study_years
    ],
    keys=study_years,
    names=["year"],
    axis=1
)

Calling method supply
Calling method supply


year                      2030                                             \
scenario                    sq    iem iem_fb diff: iem-sq diff: iemfb-iem   
carrier                                                                     
Battery Storage            4.6    4.6    4.6          0.0            -0.0   
Onshore Wind              69.0   68.9   69.1         -0.1             0.2   
gas-ccgt                  21.1   21.2   20.2          0.0            -1.0   
gas-ocgt                   0.0    0.0    0.0         -0.0             0.0   
geothermal                 0.0    0.0    0.0         -0.0            -0.0   
h2-ccgt                    1.0    1.0    1.0         -0.0             0.0   
home battery discharger    6.1    6.2    6.1          0.0            -0.1   
hydro-phs                  8.5    8.4    8.4         -0.0            -0.0   
hydro-reservoir            5.1    5.1    5.1          0.0             0.0   
load                       0.0    0.0    0.0         -0.0             0.0   
nuclear                   20.0   20.0   20.0         -0.0             0.0   
offwind-dc-fl-oh         239.1  239.2  239.0          0.1            -0.2   
solar-pv-utility          33.8   33.8   33.7         -0.0            -0.1   
solid biomass             24.9   24.9   24.9          0.0            -0.0   
waste                      0.0    0.0    0.0         -0.0             0.0   

year                      2040                                             
scenario                    sq    iem iem_fb diff: iem-sq diff: iemfb-iem  
carrier                                                                    
Battery Storage            0.6    0.6    0.7          0.0             0.1  
Onshore Wind              80.6   80.5   77.3         -0.1            -3.2  
gas-ccgt                   0.1    0.1    0.1         -0.0            -0.0  
gas-ocgt                   NaN    NaN    NaN          NaN             NaN  
geothermal                 0.0    0.0    0.0          0.0             0.0  
h2-ccgt                    6.0    5.9    6.8         -0.0             0.9  
home battery discharger  105.0  105.1  101.7          0.1            -3.5  
hydro-phs                  7.6    7.6    7.4          0.0            -0.2  
hydro-reservoir            5.0    5.0    4.9         -0.0            -0.0  
load                       0.0    0.0    0.0          0.0            -0.0  
nuclear                   42.0   42.0   42.0         -0.0            -0.0  
offwind-dc-fl-oh         462.3  462.4  456.1          0.2            -6.3  
solar-pv-utility          70.6   70.5   68.1         -0.1            -2.4  
solid biomass              8.3    8.3    8.2          0.0            -0.1  
waste                      0.0    0.0    0.0          0.0            -0.0

# Redispatch

## Constraint costs

In [120]:
pd.concat(
    [
        rc[year].constraint_costs.compare_redispatch()
            .sum() \
            .groupby("scenario").sum() \
            .div(1e6) \
            .round(1)
        for year in study_years
    ],
    keys=study_years,
    names=["year"],
    axis=1
)

Calling method constraint_costs
Calling method constraint_costs


year,2030,2040
scenario,,
diff: iem-sq,-2538.4,-7717.0
diff: iemfb-iem,49.4,-494.2
iem,807.1,11115.6
iem_fb,856.5,10621.3
sq,3345.5,18832.6


## Total redispatched energy volume
redispatch + countertrading + load shedding

In [65]:
pd.concat(
    [
        rc[year].constraint_management_volume.compare_redispatch()
            .sum() \
            .groupby("scenario").sum() \
            .div(1e6) \
            .round(1)
        for year in study_years
    ],
    keys=study_years,
    names=["year"],
    axis=1
)

Calling method constraint_management_volume
Calling method constraint_management_volume


year,2030,2040
scenario,,
diff: iem-sq,12.8,2.0
diff: iemfb-iem,-1.7,1.4
iem,331.1,464.8
iem_fb,329.4,466.2
sq,318.3,462.8


### Redispatch

#### Redispatch: ramp up

In [110]:
pd.concat(
    [
        rc[year].redispatched_volume.compare_redispatch()
            .query("carrier.str.contains('ramp up')") \
            .groupby("scenario", axis=1).sum() \
            .sum() \
            .div(1e6) \
            .round(1)
        for year in study_years
    ],
    keys=study_years,
    names=["year"],
    axis=1
)
# it is possible to show disaggregated per component or group at one of the following levels:
# per GB bus (bus), per technology-direction (ramp carrier), or total (sum):
# .groupby("bus").sum()
# .groupby("ramp carrier").sum()
# .sum()


Calling method redispatched_volume


/var/folders/9b/zpm4mfyd03s_22tcv6xfdv780000gn/T/ipykernel_61344/907413014.py:5: FutureWarning:

DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.



Calling method redispatched_volume


/var/folders/9b/zpm4mfyd03s_22tcv6xfdv780000gn/T/ipykernel_61344/907413014.py:5: FutureWarning:

DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.



year,2030,2040
scenario,,
diff: iem-sq,-4.4,-4.3
diff: iemfb-iem,0.1,-0.1
iem,146.0,167.7
iem_fb,146.1,167.6
sq,150.4,172.0


#### Redispatch: ramp down

In [111]:
pd.concat(
    [
        rc[year].redispatched_volume.compare_redispatch()
            .query("carrier.str.contains('ramp down')") \
            .groupby("scenario", axis=1).sum() \
            .sum() \
            .div(1e6) \
            .round(1)
        for year in study_years
    ],
    keys=study_years,
    names=["year"],
    axis=1
)
# it is possible to show disaggregated per component or group at one of the following levels:
# per GB bus (bus), per technology-direction (ramp carrier), or total (sum):
# .groupby("bus").sum()
# .groupby("ramp carrier").sum()
# .sum()

Calling method redispatched_volume


/var/folders/9b/zpm4mfyd03s_22tcv6xfdv780000gn/T/ipykernel_61344/647916347.py:5: FutureWarning:

DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.



Calling method redispatched_volume


/var/folders/9b/zpm4mfyd03s_22tcv6xfdv780000gn/T/ipykernel_61344/647916347.py:5: FutureWarning:

DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.



year,2030,2040
scenario,,
diff: iem-sq,3.7,0.5
diff: iemfb-iem,-0.9,-2.2
iem,161.7,229.6
iem_fb,160.8,227.3
sq,157.9,229.1


### Countertrading

#### Countertrading: ramp up

In [116]:
pd.concat(
    [
        rc[year].countertraded_volume.compare_redispatch()
            .query("carrier.str.contains('ramp up')") \
            .groupby("scenario", axis=1).sum() \
            .sum() \
            .div(1e6) \
            .round(1)
        for year in study_years
    ],
    keys=study_years,
    names=["year"],
    axis=1
)
# it is possible to show disaggregated per component or group at one of the following levels:
# per neighboring country bus (bus), interconnector name (name), or total (sum):
# .groupby("bus").sum()
# .groupby("name").sum()
# .sum()

Calling method countertraded_volume


/var/folders/9b/zpm4mfyd03s_22tcv6xfdv780000gn/T/ipykernel_61344/3195164295.py:5: FutureWarning:

DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.



Calling method countertraded_volume


/var/folders/9b/zpm4mfyd03s_22tcv6xfdv780000gn/T/ipykernel_61344/3195164295.py:5: FutureWarning:

DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.



year,2030,2040
scenario,,
diff: iem-sq,11.2,7.1
diff: iemfb-iem,-0.9,0.8
iem,19.6,64.7
iem_fb,18.6,65.5
sq,8.3,57.6


#### Countertrading: ramp down

In [117]:
pd.concat(
    [
        rc[year].countertraded_volume.compare_redispatch()
            .query("carrier.str.contains('ramp down')") \
            .groupby("scenario", axis=1).sum() \
            .sum() \
            .div(1e6) \
            .round(1)
        for year in study_years
    ],
    keys=study_years,
    names=["year"],
    axis=1
)
# it is possible to show disaggregated per component or group at one of the following levels:
# per neighboring country bus (bus), interconnector name (name), or total (sum):
# .groupby("bus").sum()
# .groupby("name").sum()
# .sum()

Calling method countertraded_volume


/var/folders/9b/zpm4mfyd03s_22tcv6xfdv780000gn/T/ipykernel_61344/2259799481.py:5: FutureWarning:

DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.



Calling method countertraded_volume


/var/folders/9b/zpm4mfyd03s_22tcv6xfdv780000gn/T/ipykernel_61344/2259799481.py:5: FutureWarning:

DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.



year,2030,2040
scenario,,
diff: iem-sq,2.6,0.5
diff: iemfb-iem,0.1,2.9
iem,3.9,2.8
iem_fb,3.9,5.8
sq,1.2,2.3


### Load shedding (ramp down only)

In [99]:
pd.concat(
    [
        rc[year].load_shedding_volume.compare_redispatch()
            .sum() \
            .groupby("scenario").sum() \
            .div(1e6) \
            .round(1)
        for year in study_years
    ],
    keys=study_years,
    names=["year"],
    axis=1
)

Calling method load_shedding_volume
Calling method load_shedding_volume


year,2030,2040
scenario,,
diff: iem-sq,-0.4,-1.8
diff: iemfb-iem,0.0,0.0
iem,0.0,0.0
iem_fb,0.0,0.0
sq,0.4,1.8


## Redispatch cost per MWh, ignoring load shedding

In [118]:
pd.concat(
    [
        rc[year].average_ramping_costs_per_mw.compare_redispatch()
            .round(1)
        for year in study_years
    ],
    keys=study_years,
    names=["year"],
    axis=1
)

Calling method average_ramping_costs_per_mw
Calling method average_ramping_costs_per_mw


year                2030                                            2040  \
scenario              sq   iem iem_fb diff: iem-sq diff: iemfb-iem    sq   
                       0     0      0            0               0     0   
avg_cost_ramp_up    14.0  11.8   11.6         -2.2            -0.2   7.1   
avg_cost_ramp_down  -2.8  -6.9   -6.3         -4.1             0.6  44.0   

year                                                          
scenario             iem iem_fb diff: iem-sq diff: iemfb-iem  
                       0      0            0               0  
avg_cost_ramp_up     4.5    4.4         -2.6            -0.1  
avg_cost_ramp_down  43.3   41.2         -0.6            -2.2